In [ ]:
import sys
sys.path.append("..")

from pynas.train.viz import plot_population_metrics, plot_best_metrics

# Plot the distribution of fitness, metric, fps, and params for each generation
f1 = "/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/Results/src/"

for folder in [f1]:
    plot_population_metrics(folder, num_generations=11, output_path=None)
    plot_best_metrics(folder, num_generations=11, output_path=None)


In [1]:
import pandas as pd

# Load the DataFrame from a .pkl file
df = pd.read_pickle("/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/Results/src/df_population_2.pkl")

# Display the DataFrame sorted by 'Fitness' in descending order
assert 'Fitness' in df.columns, "'Fitness' column not found in DataFrame"
sorted_df = df.sort_values('Metric', ascending=False)
display(sorted_df.head(1))

,Generation,Layers,Fitness,Metric,FPS,Params
13,2,"[{'layer_type': 'DenseNetBlock', 'out_channels...",1.249433,0.430979,70.350868,677320


# Test Single Model

In [2]:
from pynas.core.population import GenericUNetNetwork
import sys
sys.path.append("/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/")
from datasets.RawVessels.loader import RawVesselsDataModule
root_dir = "/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/data/TASI/DataSAR_real_refined"
dm = RawVesselsDataModule(root_dir, batch_size=4, num_workers=0, transform=None, 
                            test_size=0.15, val_size=0.15, seed=42)
dm.setup()  # Setup the data module to prepare datasets
# ================== CONFIGURE THE NETWORK ==================

parsed_layers = sorted_df.iloc[2]['Layers']
print(f"Using parsed layers: {parsed_layers}")

print(f"Input shape: {dm.input_shape}, Number of classes: {dm.num_classes}")
print(f"Number of layers: {len(parsed_layers)}")

model = GenericUNetNetwork(parsed_layers,
        input_channels=dm.input_shape[0], 
        input_height=dm.input_shape[1], 
        input_width=dm.input_shape[2], 
        num_classes=dm.num_classes,
        encoder_only=False,
)

model.eval()

/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/.venv/lib/python3.9/site-packages/lightning_fabric/__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


Using parsed layers: [{'layer_type': 'DenseNetBlock', 'out_channels_coefficient': 7, 'activation': 'ReLU'}, {'layer_type': 'MaxPool'}, {'layer_type': 'MBConv', 'expansion_factor': '6', 'activation': 'ReLU'}, {'layer_type': 'AvgPool'}, {'layer_type': 'MBConv', 'expansion_factor': '5', 'activation': 'GELU'}, {'layer_type': 'AvgPool'}, {'layer_type': 'DenseNetBlock', 'out_channels_coefficient': 4, 'activation': 'ReLU'}, {'layer_type': 'AvgPool'}, {'layer_type': 'MBConvNoRes', 'expansion_factor': '6', 'activation': 'GELU'}, {'layer_type': 'MaxPool'}, {'layer_type': 'Dropout', 'dropout_rate': 0.49}, {'layer_type': 'AvgPool'}]
Input shape: torch.Size([2, 1200, 1200]), Number of classes: 2
Number of layers: 12


GenericUNetNetwork(
  (encoder): ModuleList(
    (0): DenseNetBlock(
      (block): Sequential(
        (0): BatchNorm2d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (1): ReLU()
        (2): Conv2d(2, 14, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      )
    )
    (1): MaxPool(
      (0): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): MBConv(
      (steps): Sequential(
        (0): ConvBnAct(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1))
          (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU()
        )
        (1): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=96)
        (2): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (3): ReLU()
        (4): ConvBnAct(
          (0): Conv2d(96, 16, kernel_size=(1, 1), stride=(1, 1))
          (1): BatchNorm2d(16, eps=1e-

In [3]:
import torch
import pytorch_lightning as pl
from pynas.core.generic_lightning_module import GenericLightningSegmentationNetwork

def load_trained_weights(model: torch.nn.Module, weights_path: str) -> torch.nn.Module:
    """
    Load trained weights into the model.
    
    Args:
        model (torch.nn.Module): The model to load weights into.
        weights_path (str): Path to the saved weights file.
    
    Returns:
        torch.nn.Module: Model with loaded weights.
    """
    assert isinstance(model, torch.nn.Module), 'model must be a torch.nn.Module'
    assert isinstance(weights_path, str), 'weights_path must be a string'
    
    try:
        # Load the saved checkpoint
        checkpoint = torch.load(weights_path, map_location='cpu')
        
        # Extract state_dict from checkpoint
        if 'state_dict' in checkpoint:
            state_dict = checkpoint['state_dict']
        else:
            state_dict = checkpoint
        
        # Load the state dict into the model
        model.load_state_dict(state_dict, strict=True)
        print(f'Successfully loaded weights from {weights_path}')
        
        return model
        
    except Exception as e:
        print(f'Error loading weights from {weights_path}: {e}')
        raise

def test_model_with_exact_pynas_setup(model: torch.nn.Module, dm: pl.LightningDataModule) -> dict:
    """
    Test the model using the EXACT same setup as PyNAS training.
    
    Args:
        model (torch.nn.Module): The PyTorch model to test.
        dm (pl.LightningDataModule): The data module with correct input specifications.
    
    Returns:
        dict: Test results matching PyNAS training results.
    """
    assert isinstance(model, torch.nn.Module), 'model must be a torch.nn.Module'
    assert dm is not None, 'dm (datamodule) must not be None'
    
    print(f'Using data module input shape: {dm.input_shape}')
    print(f'Data module batch size: {dm.batch_size}')
    print(f'Number of classes: {dm.num_classes}')
    
    # CRITICAL: Use the EXACT same trainer configuration as PyNAS
    # From population.py line 978, PyNAS uses:
    trainer = pl.Trainer(
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        max_epochs=10,  # This doesn't matter for testing
        logger=False,
        enable_checkpointing=False,
    )
    
    # Create the Lightning module using the EXACT same class as PyNAS
    # Note: GenericLightningSegmentationNetwork does NOT take num_classes parameter
    lightning_module = GenericLightningSegmentationNetwork(
        model=model,
        learning_rate=1e-3,  # Same as PyNAS default
    )
    
    # IMPORTANT: Use the test method, not validate
    # PyNAS uses trainer.test() in population.py line 989
    test_results = trainer.test(lightning_module, datamodule=dm)
    print(f'Test results: {test_results}')
    
    return test_results[0] if test_results else {}

def verify_model_matches_pynas(model: torch.nn.Module, dm: pl.LightningDataModule) -> None:
    """
    Verify that the model setup matches exactly what PyNAS expects.
    
    Args:
        model (torch.nn.Module): The model to verify.
        dm (pl.LightningDataModule): The data module.
    """
    print('=== Model Verification ===')
    
    # Check model is in eval mode
    model.eval()
    print(f'Model training mode: {model.training}')
    
    # Test with actual data batch shape
    sample_batch = next(iter(dm.test_dataloader()))
    x, y = sample_batch
    print(f'Actual test batch shape: {x.shape}')
    print(f'Expected input shape: {dm.input_shape}')
    
    # Verify model can process the data
    with torch.no_grad():
        output = model(x)
        print(f'Model output shape: {output.shape}')
        print(f'Expected output classes: {dm.num_classes}')
    
    print('=== Verification Complete ===\n')

def reproduce_exact_pynas_training_setup(parsed_layers: list, dm: pl.LightningDataModule, weights_path: str = None) -> torch.nn.Module:
    """
    Recreate the model using the exact same setup as PyNAS Population.build_model().
    
    Args:
        parsed_layers (list): The parsed layers from the individual.
        dm (pl.LightningDataModule): The data module.
        weights_path (str, optional): Path to trained weights to load.
    
    Returns:
        torch.nn.Module: The model built exactly as PyNAS does.
    """
    from pynas.core.generic_unet import GenericUNetNetwork
    
    # Use the EXACT same parameters as PyNAS Population.build_model()
    model = GenericUNetNetwork(
        parsed_layers,
        input_channels=dm.input_shape[0], 
        input_height=dm.input_shape[1], 
        input_width=dm.input_shape[2], 
        num_classes=dm.num_classes,
        encoder_only=False,  # This is crucial for segmentation
    )
    
    # Load trained weights if provided
    if weights_path is not None:
        model = load_trained_weights(model, weights_path)
    
    # Set to eval mode (same as PyNAS testing)
    model.eval()
    
    return model

# Main testing code with exact PyNAS replication
print('=== Reproducing EXACT PyNAS Setup ===')

# Define the path to the trained weights
weights_path = '/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/models_traced/generation_2/model_13.pth'

# Use the correct parsed_layers for model_13.pth
parsed_layers_13 = df.loc[13, 'Layers']
assert isinstance(parsed_layers_13, list), 'parsed_layers_13 must be a list'

# Step 1: Recreate the model exactly as PyNAS does WITH trained weights
model_exact = reproduce_exact_pynas_training_setup(parsed_layers_13, dm, weights_path=weights_path)

# Step 2: Verify the setup matches PyNAS expectations
verify_model_matches_pynas(model_exact, dm)

# Step 3: Test using the exact same method as PyNAS
print('=== Testing with EXACT PyNAS Configuration ===')
exact_results = test_model_with_exact_pynas_setup(model_exact, dm)

# Step 4: Compare with your original approach
print('\n=== Comparison ===')
print('PyNAS-exact results:', exact_results)

# Additional debugging: Check if the data module setup matches PyNAS
print('\n=== Data Module Debug ===')
print(f'Train dataset size: {len(dm.train_dataloader().dataset)}')
print(f'Test dataset size: {len(dm.test_dataloader().dataset)}')
print(f'Val dataset size: {len(dm.val_dataloader().dataset)}')

# Check the exact batch that will be used for testing
test_loader = dm.test_dataloader()
sample_x, sample_y = next(iter(test_loader))
print(f'Test batch input shape: {sample_x.shape}')
print(f'Test batch target shape: {sample_y.shape}')
print(f'Input dtype: {sample_x.dtype}')
print(f'Target dtype: {sample_y.dtype}')

=== Reproducing EXACT PyNAS Setup ===
Successfully loaded weights from /Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/models_traced/generation_2/model_13.pth
=== Model Verification ===
Model training mode: False
Successfully loaded weights from /Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/models_traced/generation_2/model_13.pth
=== Model Verification ===
Model training mode: False
Actual test batch shape: torch.Size([4, 2, 1200, 1200])
Expected input shape: torch.Size([2, 1200, 1200])
Actual test batch shape: torch.Size([4, 2, 1200, 1200])
Expected input shape: torch.Size([2, 1200, 1200])


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

Model output shape: torch.Size([4, 2, 1200, 1200])
Expected output classes: 2
=== Verification Complete ===

=== Testing with EXACT PyNAS Configuration ===
Using data module input shape: torch.Size([2, 1200, 1200])
Data module batch size: 4
Number of classes: 2


/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/.venv/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing DataLoader 0:   1%|▏         | 1/73 [00:01<01:59,  0.60it/s]

/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/.venv/lib/python3.9/site-packages/pytorch_lightning/core/module.py:518: You called `self.log('test_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/.venv/lib/python3.9/site-packages/pytorch_lightning/core/module.py:518: You called `self.log('test_mse', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/.venv/lib/python3.9/site-packages/pytorch_lightning/core/module.py:518: You called `self.log('test_iou', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/.venv/lib/python3.9/site-packages/pytorch_lightning/core/module.py:518: You called `self.log('fps', ..., logger=True)` but ha

Testing DataLoader 0: 100%|██████████| 73/73 [00:23<00:00,  3.06it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│            fps            │     65.34852600097656     │
│         test_iou          │    0.43098461627960205    │
│         test_loss         │   0.0013201114488765597   │
│         test_mse          │    1.5526903867721558     │
└───────────────────────────┴───────────────────────────┘

Test results: [{'test_loss': 0.0013201114488765597, 'test_mse': 1.5526903867721558, 'test_iou': 0.43098461627960205, 'fps': 65.34852600097656}]

=== Comparison ===
PyNAS-exact results: {'test_loss': 0.0013201114488765597, 'test_mse': 1.5526903867721558, 'test_iou': 0.43098461627960205, 'fps': 65.34852600097656}

=== Data Module Debug ===
Train dataset size: 1349
Test dataset size: 289
Val dataset size: 289
Test batch input shape: torch.Size([4, 2, 1200, 1200])
Test batch target shape: torch.Size([4, 2, 1200, 1200])
Input dtype: torch.float32
Target dtype: torch.float32
Test batch input shape: torch.Size([4, 2, 1200, 1200])
Test batch target shape: torch.Size([4, 2, 1200, 1200])
Input dtype: torch.float32
Target dtype: torch.float32


In [ ]:
# Convert the value to millions
params_millions = 1984834 / 1_000_000
print(f"Parameters: {params_millions:.2f} million")

In [4]:
import torch

def export_trained_model_to_onnx(model: torch.nn.Module, input_shape: tuple, onnx_path: str) -> None:
    """
    Export a trained PyTorch model to ONNX format.
    
    Args:
        model (torch.nn.Module): The trained PyTorch model to export.
        input_shape (tuple): The shape of the input tensor (batch_size, channels, height, width).
        onnx_path (str): The file path to save the ONNX model.
    
    Returns:
        None
    """
    assert isinstance(model, torch.nn.Module), 'model must be a torch.nn.Module'
    assert isinstance(input_shape, tuple), 'input_shape must be a tuple'
    assert isinstance(onnx_path, str), 'onnx_path must be a string'
    
    # Ensure model is in eval mode
    model.eval()
    
    # Create dummy input with the correct shape and device
    device = next(model.parameters()).device
    dummy_input = torch.randn(*input_shape).to(device)
    
    # Export to ONNX
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=12,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={
            'input': {0: 'batch_size'},
            'output': {0: 'batch_size'}
        },
        verbose=False
    )
    print(f'Successfully exported trained model to {onnx_path}')

# Export the trained model to ONNX
input_shape = (1, dm.input_shape[0], dm.input_shape[1], dm.input_shape[2])  # (batch_size, channels, height, width)
onnx_path = f'trained_model_gen2_idx13.onnx'

print(f'Exporting model with input shape: {input_shape}')
print(f'Model device: {next(model_exact.parameters()).device}')

export_trained_model_to_onnx(model_exact, input_shape, onnx_path)

# Verify the exported model
try:
    import onnx
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print(f'ONNX model validation successful!')
    print(f'Input shape: {[dim.dim_value for dim in onnx_model.graph.input[0].type.tensor_type.shape.dim]}')
    print(f'Output shape: {[dim.dim_value for dim in onnx_model.graph.output[0].type.tensor_type.shape.dim]}')
except ImportError:
    print('ONNX package not available for validation, but export completed.')
except Exception as e:
    print(f'ONNX model validation failed: {e}')

Exporting model with input shape: (1, 2, 1200, 1200)
Model device: cpu


/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/pynas/core/generic_unet.py:105: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if x.shape[2:] != skip.shape[2:]:


============= Diagnostic Run torch.onnx.export version 2.0.1+cu117 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================

Successfully exported trained model to trained_model_gen2_idx13.onnx
ONNX model validation successful!
Input shape: [0, 2, 1200, 1200]
Output shape: [0, 0, 0, 0]


In [5]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import torchvision.transforms.functional as F

def save_inference_results_as_png(model: torch.nn.Module, 
                                  dm: pl.LightningDataModule, 
                                  num_samples: int = 5, 
                                  output_dir: str = 'inference_results') -> None:
    """
    Run inference on test samples and save results as PNG images.
    
    Args:
        model (torch.nn.Module): The trained model for inference.
        dm (pl.LightningDataModule): The data module containing test data.
        num_samples (int): Number of test samples to process.
        output_dir (str): Directory to save the PNG results.
    
    Returns:
        None
    """
    assert isinstance(model, torch.nn.Module), 'model must be a torch.nn.Module'
    assert dm is not None, 'dm (datamodule) must not be None'
    assert isinstance(num_samples, int) and num_samples > 0, 'num_samples must be a positive integer'
    assert isinstance(output_dir, str), 'output_dir must be a string'
    
    # Create output directory
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    # Set model to evaluation mode
    model.eval()
    device = next(model.parameters()).device
    
    # Get test dataloader
    test_loader = dm.test_dataloader()
    
    print(f'Running inference on {num_samples} test samples...')
    print(f'Saving results to: {output_path}')
    
    with torch.no_grad():
        for idx, (inputs, targets) in enumerate(test_loader):
            if idx >= num_samples:
                break
                
            # Move to device
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            # Run inference
            predictions = model(inputs)
            
            # Process each sample in the batch
            batch_size = inputs.shape[0]
            for batch_idx in range(min(batch_size, num_samples - idx)):
                sample_idx = idx * test_loader.batch_size + batch_idx
                
                # Extract single sample
                input_img = inputs[batch_idx].cpu()
                target_mask = targets[batch_idx].cpu()
                pred_mask = predictions[batch_idx].cpu()
                
                # Convert prediction to binary mask (assuming segmentation)
                if pred_mask.shape[0] > 1:  # Multi-class
                    pred_binary = torch.argmax(pred_mask, dim=0)
                else:  # Binary segmentation
                    pred_binary = torch.sigmoid(pred_mask.squeeze(0)) > 0.5
                
                # Prepare visualization
                fig, axes = plt.subplots(1, 3, figsize=(15, 5))
                
                # Original image (handle different channel configurations)
                if input_img.shape[0] == 1:  # Grayscale
                    axes[0].imshow(input_img.squeeze(0), cmap='gray')
                elif input_img.shape[0] == 3:  # RGB
                    # Normalize to [0, 1] for display
                    img_display = input_img.permute(1, 2, 0)
                    img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min())
                    axes[0].imshow(img_display)
                else:  # Multi-channel (show first channel)
                    axes[0].imshow(input_img[0], cmap='gray')
                axes[0].set_title('Input Image')
                axes[0].axis('off')
                
                # Ground truth mask
                if target_mask.dim() > 2:
                    target_display = target_mask.squeeze() if target_mask.shape[0] == 1 else torch.argmax(target_mask, dim=0)
                else:
                    target_display = target_mask
                axes[1].imshow(target_display, cmap='jet', alpha=0.7)
                axes[1].set_title('Ground Truth')
                axes[1].axis('off')
                
                # Prediction mask
                axes[2].imshow(pred_binary, cmap='jet', alpha=0.7)
                axes[2].set_title('Prediction')
                axes[2].axis('off')
                
                plt.tight_layout()
                
                # Save the figure
                output_file = output_path / f'inference_sample_{sample_idx:03d}.png'
                plt.savefig(output_file, dpi=150, bbox_inches='tight')
                plt.close()
                
                print(f'Saved: {output_file}')
                
                if sample_idx + 1 >= num_samples:
                    break
    
    print(f'Inference complete! Saved {min(num_samples, len(test_loader.dataset))} results to {output_path}')

def calculate_inference_metrics(model: torch.nn.Module, 
                              dm: pl.LightningDataModule, 
                              num_samples: int = None) -> dict:
    """
    Calculate detailed inference metrics on test data.
    
    Args:
        model (torch.nn.Module): The trained model for inference.
        dm (pl.LightningDataModule): The data module containing test data.
        num_samples (int, optional): Number of samples to evaluate. If None, uses all test data.
    
    Returns:
        dict: Dictionary containing various metrics.
    """
    assert isinstance(model, torch.nn.Module), 'model must be a torch.nn.Module'
    assert dm is not None, 'dm (datamodule) must not be None'
    
    model.eval()
    device = next(model.parameters()).device
    test_loader = dm.test_dataloader()
    
    total_samples = 0
    total_iou = 0.0
    total_accuracy = 0.0
    total_precision = 0.0
    total_recall = 0.0
    
    print('Calculating detailed inference metrics...')
    
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(test_loader):
            if num_samples is not None and total_samples >= num_samples:
                break
                
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            predictions = model(inputs)
            
            # Convert predictions to binary masks
            if predictions.shape[1] > 1:  # Multi-class
                pred_masks = torch.argmax(predictions, dim=1)
                target_masks = targets if targets.dim() == 3 else torch.argmax(targets, dim=1)
            else:  # Binary segmentation
                pred_masks = (torch.sigmoid(predictions.squeeze(1)) > 0.5).float()
                target_masks = targets.squeeze(1) if targets.dim() == 4 else targets
            
            # Calculate metrics for each sample in batch
            batch_size = inputs.shape[0]
            for i in range(batch_size):
                if num_samples is not None and total_samples >= num_samples:
                    break
                    
                pred = pred_masks[i].flatten()
                target = target_masks[i].flatten()
                
                # Calculate IoU
                intersection = (pred * target).sum()
                union = pred.sum() + target.sum() - intersection
                iou = intersection / (union + 1e-8)
                
                # Calculate accuracy
                accuracy = (pred == target).float().mean()
                
                # Calculate precision and recall
                tp = (pred * target).sum()
                fp = (pred * (1 - target)).sum()
                fn = ((1 - pred) * target).sum()
                
                precision = tp / (tp + fp + 1e-8)
                recall = tp / (tp + fn + 1e-8)
                
                total_iou += iou.item()
                total_accuracy += accuracy.item()
                total_precision += precision.item()
                total_recall += recall.item()
                total_samples += 1
    
    # Calculate averages
    avg_metrics = {
        'num_samples': total_samples,
        'avg_iou': total_iou / total_samples,
        'avg_accuracy': total_accuracy / total_samples,
        'avg_precision': total_precision / total_samples,
        'avg_recall': total_recall / total_samples,
    }
    
    # Calculate F1 score
    avg_metrics['avg_f1'] = (2 * avg_metrics['avg_precision'] * avg_metrics['avg_recall']) / \
                           (avg_metrics['avg_precision'] + avg_metrics['avg_recall'] + 1e-8)
    
    return avg_metrics

# Run inference and save results
print('=== Running Inference and Saving Results ===')

# Save inference results as PNG images
save_inference_results_as_png(
    model=model_exact,
    dm=dm,
    num_samples=10,
    output_dir='inference_results_gen2_model13'
)

# Calculate detailed metrics
inference_metrics = calculate_inference_metrics(
    model=model_exact,
    dm=dm,
    num_samples=50  # Test on 50 samples for detailed metrics
)

print('\n=== Detailed Inference Metrics ===')
for metric, value in inference_metrics.items():
    if isinstance(value, float):
        print(f'{metric}: {value:.4f}')
    else:
        print(f'{metric}: {value}')

# Save metrics to file
metrics_file = Path('inference_results_gen2_model13') / 'inference_metrics.txt'
with open(metrics_file, 'w') as f:
    f.write('Detailed Inference Metrics\n')
    f.write('=' * 30 + '\n')
    for metric, value in inference_metrics.items():
        if isinstance(value, float):
            f.write(f'{metric}: {value:.4f}\n')
        else:
            f.write(f'{metric}: {value}\n')

print(f'\nMetrics saved to: {metrics_file}')

=== Running Inference and Saving Results ===
Running inference on 10 test samples...
Saving results to: inference_results_gen2_model13
Saved: inference_results_gen2_model13/inference_sample_000.png
Saved: inference_results_gen2_model13/inference_sample_001.png
Saved: inference_results_gen2_model13/inference_sample_002.png
Saved: inference_results_gen2_model13/inference_sample_003.png
Saved: inference_results_gen2_model13/inference_sample_004.png
Saved: inference_results_gen2_model13/inference_sample_005.png
Saved: inference_results_gen2_model13/inference_sample_006.png
Saved: inference_results_gen2_model13/inference_sample_007.png
Saved: inference_results_gen2_model13/inference_sample_008.png
Saved: inference_results_gen2_model13/inference_sample_009.png
Saved: inference_results_gen2_model13/inference_sample_012.png
Saved: inference_results_gen2_model13/inference_sample_016.png
Saved: inference_results_gen2_model13/inference_sample_020.png
Saved: inference_results_gen2_model13/inferenc